# Native AHR v5 — budgeted experiment

Purpose: run the five improvements as a bounded, resumable GPU experiment and measure playing strength. Inputs: verified curriculum, native model/search, learning and arena exports; immutable protocol and successful engineering validation. Outputs: checkpoints, skill metrics, independent GNU Go development/confirmation/final results and verified local bundle.

Expected runtime: target **18 hours**, with **16 hours** allocated to learning and periodic development. A promising candidate may use the remaining budget for confirmation and 200 final games, up to **24 hours total**. These are GPU-backed wall hours, not a claim of continuous GPU kernel utilization. Clear repeated wipeouts/regression or a runtime failure stop the run early.

Checkpoint locations: `checkpoints/ahr-native-v5-main/` and `checkpoints/pipeline-ahr-native-v5/`, with separate frozen arena runs. One writer per run; checkpoint at stage boundaries and at least every ten minutes or 250 optimizer steps. Every checkpoint contains full resume state, or a controller reference to the full child state.

Exact resume: keep sources and `config/ahr_native_v5.json` unchanged. In a fresh `ahr-go` project kernel with `AHR_NATIVE_V5_RUN=1`, Run All. The original start time/deadlines are restored; resume never resets the time budget. Detached session: `tmux attach -t ahr-go-native-v5`; detach **Ctrl-b, then d**. This notebook's normal Run All validates the controller only; the launch notebook opts into the explicitly authorized long run.

GNU Go is opponent only. Bounded tactical proofs do not imply full-game optimality. Final data stays inaccessible to training; no final candidate is selected using final outcomes. Off-instance backup remains deferred by the user.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/00_instance_setup_and_recovery.ipynb').is_file())
book=nbformat.read(PROJECT/'experiments/ahr_native_v5/00_shared.ipynb',4)
for cell in book.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v5 shared','exec'),globals())
v5_load_v4(PROJECT,globals())
for name in V5_NOTEBOOKS[1:5]:v5_import_exports(PROJECT/V5_ROOT/name,globals())


## Immutable protocol and single-writer controller

In [ ]:
import io, shutil, shlex, subprocess, tarfile
from dataclasses import asdict
from zoneinfo import ZoneInfo

V5_CONTROLLER='pipeline-ahr-native-v5'
V5_TRAIN_RUN='ahr-native-v5-main'
V5_SESSION='ahr-go-native-v5'
V5_PROTOCOL='config/ahr_native_v5.json'
V5_PIPELINE_NOTEBOOK='experiments/ahr_native_v5/05_budgeted_pipeline.ipynb'
V5_TRAIN_TARGETS=(16,32,64,128,256,512,1024,2048,4096)

def v5_protocol_read(project):
    project=Path(project);protocol=v5_read_json(project/V5_PROTOCOL)
    core={k:v for k,v in protocol.items() if k!='configuration_sha256'}
    if protocol['configuration_sha256']!=v5_digest(core):raise RuntimeError('Protocol digest differs')
    if protocol['sources']!=v5_source_identity(project):raise RuntimeError('Frozen experiment sources changed')
    receipt=project/'reports/ahr_native_v5/delivery_validation.json'
    if not receipt.is_file() or file_sha256(receipt)!=protocol['validation_sha256']:raise RuntimeError('Frozen delivery validation changed')
    return protocol

def v5_protocol_prepare(project):
    project=Path(project)
    if (project/V5_PROTOCOL).exists():return v5_protocol_read(project)
    validation=v5_read_json(v5_report_path(project,'delivery_validation.json'))
    if validation.get('status')!='PASS' or validation.get('sources')!=v5_source_identity(project):raise RuntimeError('Current delivery validation required')
    for name in ('model_search','curriculum','training_recovery','arena_recovery','controller_recovery'):
        if validation['checks'].get(name)!='PASS':raise RuntimeError('Failed launch check '+name)
    if not torch.cuda.is_available() or torch.cuda.device_count()<2:raise RuntimeError('GPU 1 unavailable')
    free,total=torch.cuda.mem_get_info(1)
    if free<8*1024**3:raise RuntimeError('GPU 1 has insufficient free memory')
    training=asdict(V5TrainConfig())
    protocol={'schema':V5_SCHEMA,'controller':V5_CONTROLLER,'training_run':V5_TRAIN_RUN,'created_at_utc':v5_now(),
        'device':'cuda:1','training':training,'sources':v5_source_identity(project),
        'validation_sha256':file_sha256(v5_report_path(project,'delivery_validation.json')),
        'budget':{'requested_min_hours':12.,'requested_max_hours':24.,'target_hours':18.,'learning_window_hours':16.,'hard_limit_hours':24.,'segment_seconds':1800.,
                  'measurement':'Allocated GPU-backed experiment wall time; elapsed CUDA kernel time is not claimed to equal this budget. Connection loss does not reset the budget.'},
        'development_games':16,'final_development_games':32,'confirmation_games':32,'final_games':200,
        'training_targets':list(V5_TRAIN_TARGETS),'ahr_move_budget_s':1.,'gnu_level':1,'board_size':9,'komi':7.5,
        'scope':'Native AHR only; independently verified tactical teachers and AHR search targets; GNU is opponent only; no JEPA or external policy targets.',
        'gates':'Engineering recovery checks required. Adaptive verified-task training until measured skill/search gate passes. Final requires development and fresh confirmation parity.',
        'stop_after_unproductive_reviews':2,'unknown_policy':'Timeout/cutoff remains UNKNOWN and unscored; wins/planned conservative.',
        'off_instance_backup':'DEFERRED_BY_USER_PENDING_HUGGING_FACE_DESTINATION',
        'storage':{'source':'/dev/sda1','mount':'/','persistent_boot_disk':True,'instance_deletion_policy':'UNVERIFIED'},
        'baseline':{'v4_games':256,'wins':0,'attempts':32,'mean_score_margin':-76.25862068965517,'zero_stones_completed':26,'completed':29}}
    protocol['configuration_sha256']=v5_digest(protocol)
    with checkpoint_lock(project/'checkpoints'/'ahr-native-v5-protocol'):
        if (project/V5_PROTOCOL).exists():return v5_protocol_read(project)
        v5_atomic_json(project/V5_PROTOCOL,protocol)
    return protocol

def v5_controller_load(project,protocol,run_id=V5_CONTROLLER):
    directory=Path(project)/'checkpoints'/run_id;manifest_path=directory/'manifest.json'
    if not manifest_path.exists():
        if list(directory.glob('*.pt')):raise RuntimeError('Controller payload without manifest')
        return None
    manifest=v5_read_json(manifest_path);entries=manifest.get('entries',[])
    if not entries or manifest.get('latest')!=max(entries,key=lambda e:e['global_step'])['file']:raise RuntimeError('Invalid controller latest pointer')
    if manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Missing controller best pointer')
    loaded={}
    for entry in entries:
        state=_checkpoint_entry_state(directory,entry)
        if state is None or state.get('schema')!=V5_SCHEMA or state.get('config')!=protocol:raise RuntimeError('Corrupt or changed controller identity; no restart permitted')
        loaded[entry['file']]=state
    state=loaded[manifest['latest']]
    if state['sampler']['phase'] not in ('learning','development','confirmation','final','bundle','complete'):raise RuntimeError('Invalid controller phase')
    restore_rng(state['rng']);return state

def v5_controller_initial(protocol):
    now=time.time()
    return {'schema':V5_SCHEMA,'config':copy.deepcopy(protocol),'global_step':0,'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':capture_rng(),
        'sampler':{'phase':'learning','target_index':0,'review_index':0},
        'metrics':{'status':'RUNNING','started_unix':now,'learning_deadline_unix':now+3600*protocol['budget']['learning_window_hours'],
                   'target_deadline_unix':now+3600*protocol['budget']['target_hours'],'hard_deadline_unix':now+3600*protocol['budget']['hard_limit_hours'],
                   'development':[],'confirmation':None,'final':None,'best_development':None,'last_error':None,'unproductive_reviews':0,'last_review_unix':now},
        'child_state_contract':'Model/optimizer/scaler/RNG/sampler/replay/restart/ongoing Go game state is committed in each child run; controller commits phase and unchanged wall budget.'}

def v5_controller_commit(project,state,run_id=V5_CONTROLLER,best=False):
    state['global_step']+=1;state['rng']=capture_rng()
    entry=_save_checkpoint_unlocked(Path(project)/'checkpoints'/run_id,state,best=best,resume_cell=V5_PIPELINE_NOTEBOOK+': unchanged sources/config; AHR_NATIVE_V5_RUN=1; Run All')
    m=state['metrics'];phase=state['sampler']['phase']
    report={'schema':V5_SCHEMA,'status':m['status'],'phase':phase,'training_run':state['config']['training_run'],'session':V5_SESSION,
        'global_step':state['global_step'],'target_games':state['config']['training_targets'][state['sampler']['target_index']],
        'started_unix':m['started_unix'],'elapsed_hours':(time.time()-m['started_unix'])/3600,'learning_deadline_unix':m['learning_deadline_unix'],
        'target_deadline_unix':m['target_deadline_unix'],'hard_deadline_unix':m['hard_deadline_unix'],
        'development':m['development'],'best_development':m['best_development'],'confirmation':m['confirmation'],'final':m['final'],
        'training':m.get('training'),'last_error':m['last_error'],'checkpoint':entry,'updated_at_utc':v5_now()}
    if run_id==V5_CONTROLLER:
        v5_atomic_json(v5_report_path(project,'pipeline_status.json'),report)
        print(json.dumps({k:report[k] for k in ('status','phase','target_games','elapsed_hours','updated_at_utc')}),flush=True)
    return entry

def v5_development_rank(receipt):
    margin=receipt.get('mean_score_margin');return (receipt.get('wins_over_planned',0),margin if margin is not None else -1000)

def v5_receipt_wipeout(project,receipt):
    path=Path(project)/receipt['report_path']
    report=v5_read_json(path);games=report.get('failure_analysis',{}).get('games',[])
    completed=[g for g in games if g.get('status')=='COMPLETE']
    return sum(g.get('surviving_candidate_stones')==0 for g in completed)/len(completed) if completed else None

def v5_local_bundle(project,state):
    project=Path(project);out=project/'exports/bundles';out.mkdir(parents=True,exist_ok=True)
    stamp=v5_digest({'controller_step':state['global_step'],'protocol':state['config']['configuration_sha256']})[:12]
    target=out/('ahr-native-v5-'+stamp+'.tar.gz')
    roots=[project/'notebooks',project/V5_ROOT,project/'config',project/'reports/ahr_native_v5',project/'checkpoints'/V5_CONTROLLER,
           project/'checkpoints'/V5_TRAIN_RUN,project/'checkpoints'/'ahr-v5-curriculum-generation',project/'checkpoints'/'ahr-v3-B-s01',
           project/'experiments/ahr_native_v4/00_shared.ipynb',project/'experiments/ahr_native_v4/01_native_ahr_and_tactics.ipynb',
           project/'experiments/ahr_native_v4/02_competitive_training.ipynb',project/'experiments/ahr_native_v4/03_gnugo_evaluation.ipynb',
           project/'experiments/ahr_native_v4/04_pipeline.ipynb',project/'experiments/ahr_native_v4/data/trajectories/ahr-native-v4-main',
           project/'experiments/ahr_jepa_v3/data/curriculum.json',project/'experiments/ahr_jepa_v3/data/curriculum.manifest.json',
           project/'.venv/third_party/gnugo/usr/bin/gnugo',project/'AHR_Unified_Theory (1).pdf']
    for item in state['metrics']['development']+[r for r in (state['metrics'].get('confirmation'),state['metrics'].get('final')) if r]:
        roots.append(project/'checkpoints'/item['eval_run'])
    active_review=state['metrics'].get('review_eval_run')
    if active_review:roots.append(project/'checkpoints'/active_review)
    files=set()
    for root in roots:
        if root.is_file():files.add(root)
        elif root.is_dir():files.update(p for p in root.rglob('*') if p.is_file() and not p.is_symlink() and p.name!='.checkpoint.lock' and not p.name.endswith('.tmp'))
    manifest={str(p.relative_to(project)):file_sha256(p) for p in sorted(files)}
    temporary=target.with_name('.'+target.name+'.tmp')
    with tarfile.open(temporary,'w:gz',compresslevel=3) as archive:
        for p in sorted(files):archive.add(p,arcname=str(p.relative_to(project)),recursive=False)
        data=(json.dumps(manifest,sort_keys=True,indent=2)+'\n').encode();info=tarfile.TarInfo('bundle_hashes.json');info.size=len(data);archive.addfile(info,io.BytesIO(data))
    os.replace(temporary,target)
    with tarfile.open(target,'r:gz') as archive:
        for name,digest in manifest.items():
            stream=archive.extractfile(name)
            if stream is None or hashlib.sha256(stream.read()).hexdigest()!=digest:raise RuntimeError('Backup member hash differs: '+name)
    receipt={'status':'VERIFIED_LOCAL','bundle':str(target.relative_to(project)),'sha256':file_sha256(target),'bytes':target.stat().st_size,
        'members':len(manifest),'off_instance_backup':state['config']['off_instance_backup'],'restore':'Extract with tarfile filter=data; verify bundle_hashes.json; recreate recorded dependencies; exact checkpoint resume with unchanged source/config.'}
    v5_atomic_json(v5_report_path(project,'local_bundle.json'),receipt);return receipt

def v5_run_budgeted_pipeline(project):
    project=Path(project);protocol=v5_protocol_read(project)
    with checkpoint_lock(project/'checkpoints'/V5_CONTROLLER):
        state=v5_controller_load(project,protocol)
        if state is None:state=v5_controller_initial(protocol);v5_controller_commit(project,state,best=True)
        m=state['metrics'];sampler=state['sampler']
        if sampler['phase']!='complete' and m['status']=='INTERRUPTED_OR_FAILED':m['status']='RUNNING'
        try:
            while sampler['phase']!='complete':
                phase=sampler['phase'];now=time.time();m['last_error']=None
                if now>=m['hard_deadline_unix'] and phase not in ('bundle','complete'):
                    m['status']='HARD_BUDGET_REACHED';sampler['phase']='bundle';v5_controller_commit(project,state);continue
                if phase=='learning':
                    target=protocol['training_targets'][sampler['target_index']]
                    deadline=min(m['learning_deadline_unix'],time.time()+protocol['budget']['segment_seconds'])
                    if time.time()<deadline:
                        m['training']=v5_run_learning(project,V5_TRAIN_RUN,V5TrainConfig(**protocol['training']),run_until_games=target,deadline_unix=deadline)
                    training=m.get('training',{});games=training.get('games_finished',0)
                    ready=games>=target;due=time.time()-m['last_review_unix']>=4*3600
                    end=time.time()>=m['learning_deadline_unix'] or games>=protocol['training']['scheduled_games']
                    if ready or due or end:
                        m['review_terminal']=end;m['review_games']=protocol['final_development_games'] if end else protocol['development_games']
                        m['review_eval_run']=f'ahr-native-v5-dev-r{sampler["review_index"]:03d}-g{games:04d}'
                        sampler['phase']='development'
                elif phase=='development':
                    deadline=m['hard_deadline_unix'] if m.get('review_terminal') else min(m['learning_deadline_unix'],time.time()+1800)
                    r=v5_run_gnu_evaluation(project,V5_TRAIN_RUN,m['review_eval_run'],split='development',games=m['review_games'],device=protocol['device'],deadline_unix=deadline)
                    if r['status']!='COMPLETE':
                        if time.time()>=m['learning_deadline_unix']:m['review_terminal']=True
                        if time.time()>=m['hard_deadline_unix']:m['status']='HARD_BUDGET_REACHED';sampler['phase']='bundle'
                        v5_controller_commit(project,state);continue
                    old=m['best_development'];m['development'].append(r);m['last_review_unix']=time.time();sampler['review_index']+=1
                    if old is None or v5_development_rank(r)>v5_development_rank(old):m['best_development']=r
                    wipeout=v5_receipt_wipeout(project,r);r['wipeout_fraction_completed']=wipeout
                    poor=(r['wins']==0 and wipeout is not None and wipeout>=.8 and old is not None and v5_development_rank(r)<=v5_development_rank(old))
                    m['unproductive_reviews']=m['unproductive_reviews']+1 if poor else 0
                    if r.get('observed_half_wins',False):
                        m['confirmation_candidate']=r;m['confirmation_eval_run']=f'ahr-native-v5-confirm-r{sampler["review_index"]:03d}';sampler['phase']='confirmation'
                    elif m['unproductive_reviews']>=protocol['stop_after_unproductive_reviews'] and time.time()-m['started_unix']>=protocol['budget']['requested_min_hours']*3600:
                        m['status']='STOPPED_UNPRODUCTIVE';m['final_readiness']='FAIL';sampler['phase']='bundle'
                    elif m.get('review_terminal') or time.time()>=m['learning_deadline_unix']:
                        m['status']='BUDGET_COMPLETE_FINAL_NOT_READY';m['final_readiness']='FAIL';sampler['phase']='bundle'
                    else:
                        games=m.get('training',{}).get('games_finished',0)
                        while sampler['target_index']+1<len(protocol['training_targets']) and protocol['training_targets'][sampler['target_index']]<=games:sampler['target_index']+=1
                        sampler['phase']='learning'
                elif phase=='confirmation':
                    c=m['confirmation_candidate'];r=v5_run_gnu_evaluation(project,V5_TRAIN_RUN,m['confirmation_eval_run'],split='confirmation',games=protocol['confirmation_games'],checkpoint_path=c['checkpoint_path'],device=protocol['device'],deadline_unix=m['hard_deadline_unix'])
                    m['confirmation']=r
                    if r['status']!='COMPLETE':
                        if time.time()>=m['hard_deadline_unix']:m['status']='HARD_BUDGET_REACHED';sampler['phase']='bundle'
                    elif r.get('observed_half_wins',False):
                        m['final_readiness']='PASS';sampler['phase']='final'
                    elif time.time()>=m['learning_deadline_unix'] or m.get('review_terminal'):
                        m['status']='BUDGET_COMPLETE_FINAL_NOT_READY';m['final_readiness']='FAIL';sampler['phase']='bundle'
                    else:
                        # A failed fresh confirmation requires further training, not new seeds on the same milestone.
                        games=m.get('training',{}).get('games_finished',0)
                        while sampler['target_index']+1<len(protocol['training_targets']) and protocol['training_targets'][sampler['target_index']]<=games:sampler['target_index']+=1
                        sampler['phase']='learning'
                elif phase=='final':
                    c=m['confirmation'];r=v5_run_gnu_evaluation(project,V5_TRAIN_RUN,'ahr-native-v5-final-200',split='final',games=200,checkpoint_path=c['checkpoint_path'],device=protocol['device'],deadline_unix=m['hard_deadline_unix'])
                    m['final']=r
                    if r['status']=='COMPLETE':m['status']='TARGET_OBSERVED' if r['observed_half_wins'] else 'FINAL_COMPLETE_TARGET_NOT_REACHED';sampler['phase']='bundle'
                    elif time.time()>=m['hard_deadline_unix']:m['status']='HARD_BUDGET_REACHED_FINAL_INCOMPLETE';sampler['phase']='bundle'
                elif phase=='bundle':
                    if m['status']=='RUNNING':m['status']='BUDGET_COMPLETE'
                    v5_controller_commit(project,state)
                    m['local_bundle']=v5_local_bundle(project,state);sampler['phase']='complete'
                    completion={'schema':V5_SCHEMA,'status':m['status'],'elapsed_hours':(time.time()-m['started_unix'])/3600,
                        'training':m.get('training'),'development':m['development'],'confirmation':m['confirmation'],'final':m['final'],
                        'final_readiness':m.get('final_readiness','NOT_MEASURED'),'all_five_changes_implemented':True,
                        'target_observed':bool(m.get('final') and m['final'].get('observed_half_wins')),'local_bundle':m['local_bundle'],'completed_at_utc':v5_now()}
                    v5_atomic_json(v5_report_path(project,'completion.json'),completion)
                else:raise RuntimeError('Unknown controller phase '+phase)
                v5_controller_commit(project,state,best=sampler['phase']=='complete')
        except BaseException as error:
            m['status']='INTERRUPTED_OR_FAILED';m['last_error']=type(error).__name__+': '+str(error)
            v5_controller_commit(project,state);raise
    return v5_read_json(v5_report_path(project,'pipeline_status.json'))

def v5_validate_controller_recovery(project):
    project=Path(project);run='ahr-native-v5-controller-smoke-'+uuid.uuid4().hex[:10]
    protocol={'schema':V5_SCHEMA,'training_run':'fixture-no-training','sources':v5_source_identity(project),
        'training_targets':[16],'budget':{'learning_window_hours':16.,'target_hours':18.,'hard_limit_hours':24.}}
    directory=project/'checkpoints'/run
    with checkpoint_lock(directory):
        state=v5_controller_initial(protocol)
        v5_controller_commit(project,state,run,best=True)
        state['sampler']['phase']='development';state['metrics']['fixture_child_step']=71
        v5_controller_commit(project,state,run);v5_controller_commit(project,state,run)
        restored=v5_controller_load(project,protocol,run)
        if restored['global_step']!=3 or restored['metrics']['fixture_child_step']!=71:raise AssertionError('Controller step/phase recovery differs')
        if restored['metrics']['started_unix']!=state['metrics']['started_unix']:raise AssertionError('Resume reset wall budget')
        manifest=v5_read_json(directory/'manifest.json')
        if len(manifest['entries'])<2:raise AssertionError('Latest two retention failed')
        latest=next(e for e in manifest['entries'] if e['file']==manifest['latest']);payload=directory/latest['file'];original=payload.read_bytes()
        payload.write_bytes(original+b'corrupt')
        try:
            try:v5_controller_load(project,protocol,run)
            except RuntimeError:pass
            else:raise AssertionError('Corruption accepted')
        finally:payload.write_bytes(original)
        changed=copy.deepcopy(protocol);changed['sources']['fixture']='changed'
        try:v5_controller_load(project,changed,run)
        except RuntimeError:pass
        else:raise AssertionError('Changed controller identity accepted')
    receipt={'status':'PASS','sources':v5_source_identity(project),'checks':{'latest2_best':True,'phase_child_step_restore':True,'budget_not_reset':True,'corruption_rejected':True,'changed_identity_rejected':True},'fixture_run':run,'validated_at_utc':v5_now()}
    v5_atomic_json(v5_report_path(project,'controller_recovery.json'),receipt);return receipt

def v5_launch_detached(project):
    project=Path(project);protocol=v5_protocol_read(project)
    with checkpoint_lock(project/'checkpoints'/'ahr-native-v5-launch'):
        if subprocess.run(['tmux','has-session','-t','='+V5_SESSION],capture_output=True).returncode==0:raise RuntimeError('AHR v5 session already exists; inspect single writer')
        command=['env','AHR_NATIVE_V5_RUN=1','CUBLAS_WORKSPACE_CONFIG=:4096:8',sys.executable,'-u','-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(project/V5_PIPELINE_NOTEBOOK)]
        log=v5_report_path(project,'controller.log')
        subprocess.run(['tmux','new-session','-d','-s',V5_SESSION,'-n','pipeline','-c',str(project),'exec '+shlex.join(command)+' >> '+shlex.quote(str(log))+' 2>&1'],check=True)
        subprocess.run(['tmux','set-window-option','-t',V5_SESSION+':pipeline','remain-on-exit','on'],check=True)
        receipt={'status':'LAUNCHED','session':V5_SESSION,'configuration_sha256':protocol['configuration_sha256'],'device':'cuda:1',
            'attach':'tmux attach -t '+V5_SESSION,'detach':'Ctrl-b, then d','log':str(log.relative_to(project)),
            'target_hours':18,'hard_limit_hours':24,'resume':'Unchanged configuration/sources; AHR_NATIVE_V5_RUN=1; Run All in05_budgeted_pipeline.ipynb. Original start/deadlines are restored.',
            'launched_at_utc':v5_now()}
        v5_atomic_json(v5_report_path(project,'launch.json'),receipt);return receipt


## Recovery validation or authorized main run

In [ ]:
if os.environ.get('AHR_NATIVE_V5_RUN')=='1':
    result=v5_run_budgeted_pipeline(PROJECT)
else:
    result=v5_validate_controller_recovery(PROJECT)
print({k:result[k] for k in ('status','phase','global_step') if k in result})